# DA6401 Assignment 2 — Training Notebook

## Strategy
```
Drive (permanent storage)  →  copy to /content (fast local SSD)  →  train
                                                                        ↓
                                              save checkpoints to Drive
```
**Always run Cell 0 first at the start of every session.**

⚠️ Set GPU before starting: `Runtime → Change runtime type → T4 GPU`

---
## CELL 0 — Session Setup (run this FIRST every session)

In [ ]:
# ── 1. Mount Drive ───────────────────────────────────────────
import os
os.system('fusermount -u /content/drive 2>/dev/null || true')

from google.colab import drive
drive.mount('/content/drive')

# ── 2. Paths ─────────────────────────────────────────────────
import sys

DRIVE_ROOT   = '/content/drive/MyDrive/DA6401_Assignment2'
PROJECT_ROOT = f'{DRIVE_ROOT}/code'
DRIVE_CKPT   = f'{DRIVE_ROOT}/checkpoints'
LOCAL_DATA   = '/content/pets'          # fast local SSD
ZIP_PATH     = f'{DRIVE_ROOT}/pets_dataset.zip'

os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
%cd {PROJECT_ROOT}

# ── 3. Install packages ───────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q', 'albumentations', 'wandb', 'scikit-learn'],
               check=True)

# ── 4. GPU check ──────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU — go to Runtime → Change runtime type → T4 GPU')

# ── 5. Checkpoint status ──────────────────────────────────────
print('\nCheckpoint status:')
for fname in ['classifier.pth', 'localizer.pth', 'unet.pth']:
    fp = os.path.join(DRIVE_CKPT, fname)
    if os.path.exists(fp):
        c = torch.load(fp, map_location='cpu')
        print(f'  ✅  {fname:<22} epoch={c.get("epoch")}  metric={c.get("best_metric",0):.4f}')
    else:
        print(f'  ❌  {fname:<22} not trained yet')

print('\nSetup complete!')

Mounted at /content/drive
/content/drive/MyDrive/DA6401_Assignment2/code
GPU  : Tesla T4
VRAM : 15.6 GB

Checkpoint status:
  ✅  classifier.pth         epoch=23  metric=0.8800
  ✅  localizer.pth          epoch=35  metric=0.6352
  ❌  unet.pth               not trained yet

Setup complete!


In [ ]:
import os

# Download full dataset fresh to a temp location
print("Downloading full dataset...")
os.system('wget -q --show-progress -O /content/images.tar.gz https://thor.robots.ox.ac.uk/pets/images.tar.gz')
os.system('wget -q --show-progress -O /content/annots.tar.gz https://thor.robots.ox.ac.uk/pets/annotations.tar.gz')

print("Extracting...")
os.system('tar -xzf /content/images.tar.gz -C /content/pets/')
os.system('tar -xzf /content/annots.tar.gz -C /content/pets/')

# Cleanup tars
os.system('rm /content/images.tar.gz /content/annots.tar.gz')

# Verify
n = len(os.listdir('/content/pets/images'))
print(f"Total images now: {n}")

Extracting...
Total images now: 7393


In [ ]:
# Run this AFTER download completes
print("Creating complete zip for Drive...")
os.system('cd /content && zip -r /content/drive/MyDrive/DA6401_Assignment2/pets_dataset.zip pets/')
print("Done! Complete zip saved to Drive.")

Creating complete zip for Drive...
Done! Complete zip saved to Drive.


---
## CELL 2 — Copy & extract dataset to local SSD (run every session)
This is the key step — copies from Drive to fast local `/content/pets`.
Takes ~2 minutes. Training will be 3-5x faster after this.

In [ ]:
import os, time

DRIVE_ROOT = '/content/drive/MyDrive/DA6401_Assignment2'
ZIP_PATH   = f'{DRIVE_ROOT}/pets_dataset.zip'
LOCAL_DATA = '/content/pets'

# Check if already extracted this session
if os.path.exists(os.path.join(LOCAL_DATA, 'images')):
    n = len(os.listdir(os.path.join(LOCAL_DATA, 'images')))
    print(f'Dataset already extracted locally ({n} images) — skipping.')
else:
    os.makedirs(LOCAL_DATA, exist_ok=True)

    if os.path.exists(ZIP_PATH):
        print('Copying ZIP from Drive to local ...')
        t0 = time.time()
        os.system(f'cp "{ZIP_PATH}" /content/pets_dataset.zip')
        print(f'Copy done in {time.time()-t0:.0f}s')

        print('Extracting ...')
        t1 = time.time()
        os.system('unzip -q /content/pets_dataset.zip -d /content/tmp_extract')

        # Handle nested folder structure from zip
        inner = '/content/tmp_extract/dataset'
        if os.path.exists(inner):
            os.system(f'mv {inner}/* {LOCAL_DATA}/')
        else:
            os.system(f'mv /content/tmp_extract/* {LOCAL_DATA}/')
        os.system('rm -rf /content/tmp_extract /content/pets_dataset.zip')
        print(f'Extraction done in {time.time()-t1:.0f}s')
    else:
        # ZIP not found — download fresh
        print('ZIP not found in Drive. Downloading dataset fresh ...')
        os.system(f'wget -q --show-progress -O /content/images.tar.gz https://thor.robots.ox.ac.uk/pets/images.tar.gz')
        os.system(f'wget -q --show-progress -O /content/annots.tar.gz  https://thor.robots.ox.ac.uk/pets/annotations.tar.gz')
        os.system(f'tar -xzf /content/images.tar.gz -C {LOCAL_DATA}')
        os.system(f'tar -xzf /content/annots.tar.gz -C {LOCAL_DATA}')
        os.system('rm /content/images.tar.gz /content/annots.tar.gz')
        print('Download + extraction complete.')

    # Verify
    n_imgs  = len(os.listdir(os.path.join(LOCAL_DATA, 'images')))
    n_masks = len(os.listdir(os.path.join(LOCAL_DATA, 'annotations', 'trimaps')))
    n_xmls  = len(os.listdir(os.path.join(LOCAL_DATA, 'annotations', 'xmls')))
    print(f'\nImages : {n_imgs}')
    print(f'Masks  : {n_masks}')
    print(f'XMLs   : {n_xmls}')
    print('Dataset ready locally at /content/pets ✅')

Copying ZIP from Drive to local ...
Copy done in 34s
Extracting ...
Extraction done in 21s

Images : 7393
Masks  : 14780
XMLs   : 3686
Dataset ready locally at /content/pets ✅


---
## CELL 3 — Sanity check

In [ ]:
import torch
from models.vgg11 import VGG11Encoder
from models.layers import CustomDropout
from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from models.segmentation import VGG11UNet
from losses.iou_loss import IoULoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dummy  = torch.randn(2, 3, 224, 224).to(device)
passed, failed = 0, 0

def chk(name, cond, detail=''):
    global passed, failed
    if cond:
        print(f'  ✅  {name}  {detail}')
        passed += 1
    else:
        print(f'  ❌  {name}  {detail}')
        failed += 1

enc = VGG11Encoder().to(device)
bn, skips = enc(dummy, return_features=True)
chk('Encoder bottleneck', bn.shape == torch.Size([2,512,7,7]),    str(bn.shape))
chk('Skip s1',            skips['s1'].shape == torch.Size([2,64,112,112]))
chk('Skip s4',            skips['s4'].shape == torch.Size([2,512,14,14]))

clf = VGG11Classifier(num_classes=37).to(device)
chk('Classifier output',  clf(dummy).shape == torch.Size([2,37]))

loc = VGG11Localizer(freeze_backbone=False).to(device)
out_l = loc(dummy)
chk('Localizer output',   out_l.shape == torch.Size([2,4]))
chk('Bbox in [0,224]',    0 <= out_l.min().item() and out_l.max().item() <= 224)

seg = VGG11UNet(num_classes=3).to(device)
chk('UNet output',        seg(dummy).shape == torch.Size([2,3,224,224]))

iou = IoULoss(reduction='mean')
pb = torch.tensor([[112.,112.,80.,80.]])
gb = torch.tensor([[112.,112.,80.,80.]])
chk('IoU loss perfect',   iou(pb,gb).item() < 1e-4,  f'{iou(pb,gb).item():.6f}')
pb2 = torch.tensor([[0.,0.,5.,5.]])
gb2 = torch.tensor([[220.,220.,5.,5.]])
chk('IoU loss no overlap', iou(pb2,gb2).item() > 0.99, f'{iou(pb2,gb2).item():.6f}')

drop = CustomDropout(p=0.5)
drop.train()
t = torch.ones(10000)
frac = (drop(t)==0).float().mean().item()
chk('Dropout train ~50%', 0.45 < frac < 0.55, f'{frac:.3f}')
drop.eval()
chk('Dropout eval identity', (drop(t)==t).all().item())

print(f'\n{passed} passed, {failed} failed')
if failed == 0:
    print('All checks passed ✅ — safe to train!')

  ✅  Encoder bottleneck  torch.Size([2, 512, 7, 7])
  ✅  Skip s1  
  ✅  Skip s4  
  → Pretrained ImageNet weights loaded into VGG11Encoder
  ✅  Classifier output  
  ✅  Localizer output  
  ✅  Bbox in [0,224]  
  ✅  UNet output  
  ✅  IoU loss perfect  0.000000
  ✅  IoU loss no overlap  1.000000
  ✅  Dropout train ~50%  0.495
  ✅  Dropout eval identity  

11 passed, 0 failed
All checks passed ✅ — safe to train!


---
## CELL 4 — W&B Login

In [ ]:
import os, wandb
# Paste your W&B API key below
WANDB_KEY = "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"
os.environ["WANDB_API_KEY"] = WANDB_KEY
wandb.login(key=WANDB_KEY, relogin=True)
print('W&B ready!')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: da25m019 (da25m019-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B ready!


---
## CELL 5 — TASK 1: Train Classifier
⏱ ~1.5-2 hrs on T4  
Saves: `checkpoints/classifier.pth` to Drive

In [ ]:
!python train.py \
    --task classification \
    --data_root /content/pets \
    --ckpt_dir /content/drive/MyDrive/DA6401_Assignment2/checkpoints \
    --epochs 30 \
    --batch_size 64 \
    --lr 3e-4 \
    --dropout_p 0.4

  TASK 1 — Classification  (pretrained backbone)
/content/drive/MyDrive/DA6401_Assignment2/code/data/pets_dataset.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
  → Pretrained ImageNet weights loaded into VGG11Encoder
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: da25m019 (da25m019-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ setting up run k2f5rncu (0.1s)
wandb: ⣽ setting up run k2f5rncu (0.1s)
wandb: ⣾ setting up run k2f5rncu (0.1s)
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /content/drive/MyDrive/DA6401_Assignment2/code/wandb/run-20260406_163153-k2f5rncu
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run clas

In [ ]:
# Verify
import torch, os
p = '/content/drive/MyDrive/DA6401_Assignment2/checkpoints/classifier.pth'
if os.path.exists(p):
    c = torch.load(p, map_location='cpu')
    print(f'classifier.pth ✅  epoch={c["epoch"]}  best_f1={c["best_metric"]:.4f}')
else:
    print('classifier.pth not found ❌')

classifier.pth ✅  epoch=23  best_f1=0.8800


---
## CELL 6 — TASK 2: Train Localizer
⏱ ~45 mins on T4  
Requires: `classifier.pth`  
Saves: `checkpoints/localizer.pth`

In [ ]:
!python train.py \
    --task localization \
    --data_root /content/pets \
    --ckpt_dir  /content/drive/MyDrive/DA6401_Assignment2/checkpoints \
    --epochs    40 \
    --batch_size 32 \
    --lr        1e-4 \
    --dropout_p 0.4

  TASK 2 — Localisation
/content/drive/MyDrive/DA6401_Assignment2/code/data/pets_dataset.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
  → Backbone loaded from classifier.pth
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: da25m019 (da25m019-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ setting up run 0f2bnv1m (0.1s)
wandb: ⣽ setting up run 0f2bnv1m (0.1s)
wandb: ⣾ setting up run 0f2bnv1m (0.1s)
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /content/drive/MyDrive/DA6401_Assignment2/code/wandb/run-20260406_171835-0f2bnv1m
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run localizer
wandb: ⭐️ View project at https://wa

In [ ]:
import torch, os
p = '/content/drive/MyDrive/DA6401_Assignment2/checkpoints/localizer.pth'
if os.path.exists(p):
    c = torch.load(p, map_location='cpu')
    print(f'localizer.pth ✅  epoch={c["epoch"]}  best_iou={c["best_metric"]:.4f}')
else:
    print('localizer.pth not found ❌')

localizer.pth ✅  epoch=35  best_iou=0.6352


---
## CELL 7 — TASK 3a: U-Net Frozen Encoder
⏱ ~45 mins  
Saves: `checkpoints/unet_frozen.pth`

In [ ]:
!python train.py \
    --task segmentation \
    --unet_mode frozen \
    --data_root /content/pets \
    --ckpt_dir  /content/drive/MyDrive/DA6401_Assignment2/checkpoints \
    --epochs    30 \
    --batch_size 16 \
    --lr        1e-4 \
    --dropout_p 0.4

  TASK 3 — Segmentation  [frozen]
/content/drive/MyDrive/DA6401_Assignment2/code/data/pets_dataset.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
  → Encoder loaded from classifier.pth
  → Encoder fully frozen
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: da25m019 (da25m019-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ setting up run o8erg3m1 (0.1s)
wandb: ⣽ setting up run o8erg3m1 (0.1s)
wandb: ⣾ setting up run o8erg3m1 (0.1s)
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /content/drive/MyDrive/DA6401_Assignment2/code/wandb/run-20260406_173925-o8erg3m1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run unet-frozen


## CELL 8 — TASK 3b: U-Net Partial Fine-tune
⏱ ~50 mins  
Saves: `checkpoints/unet_partial.pth`

In [ ]:
!python train.py \
    --task segmentation \
    --unet_mode partial \
    --data_root /content/pets \
    --ckpt_dir  /content/drive/MyDrive/DA6401_Assignment2/checkpoints \
    --epochs    30 \
    --batch_size 16 \
    --lr        1e-4 \
    --dropout_p 0.4

  TASK 3 — Segmentation  [partial]
/content/drive/MyDrive/DA6401_Assignment2/code/data/pets_dataset.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
  → Encoder loaded from classifier.pth
  → Encoder partially frozen (stage1+2 frozen, stage3-5 trainable)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: da25m019 (da25m019-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ setting up run ykuqc7lo (0.3s)
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /content/drive/MyDrive/DA6401_Assignment2/code/wandb/run-20260407_115421-ykuqc7lo
wandb: Run `wandb offline` to turn off syn

## CELL 9 — TASK 3c: U-Net Full Fine-tune ← FINAL unet.pth
⏱ ~55 mins  
Saves: `checkpoints/unet.pth`  (this is the one submitted)

In [ ]:
!python train.py \
    --task segmentation \
    --unet_mode full \
    --data_root /content/pets \
    --ckpt_dir  /content/drive/MyDrive/DA6401_Assignment2/checkpoints \
    --epochs    30 \
    --batch_size 16 \
    --lr        1e-4 \
    --dropout_p 0.4

  TASK 3 — Segmentation  [full]
/content/drive/MyDrive/DA6401_Assignment2/code/data/pets_dataset.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
  → Encoder loaded from classifier.pth
  → Full fine-tuning — all weights trainable
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: da25m019 (da25m019-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
]11;?]11;?wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ setting up run a19t2hj7 (0.3s)
wandb: ⣾ setting up run a19t2hj7 (0.3s)
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /content/drive/MyDrive/DA6401_Assignment2/code/wandb/run-20260407_122725-a19t2hj7
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing r

---
## CELL 10 — Final checkpoint check

In [ ]:
import os, torch
CKPT = '/content/drive/MyDrive/DA6401_Assignment2/checkpoints'
print('='*52)
print('FINAL CHECKPOINT STATUS')
print('='*52)
all_ok = True
for fname in ['classifier.pth', 'localizer.pth',
              'unet_frozen.pth', 'unet_partial.pth', 'unet.pth']:
    fp = os.path.join(CKPT, fname)
    if os.path.exists(fp):
        c  = torch.load(fp, map_location='cpu')
        mb = os.path.getsize(fp)/1e6
        print(f'  ✅  {fname:<22} epoch={c.get("epoch")}  '
              f'metric={c.get("best_metric",0):.4f}  {mb:.0f}MB')
    else:
        print(f'  ❌  {fname:<22} NOT FOUND')
        all_ok = False
print()
if all_ok:
    print('All checkpoints ready — download the 3 submission files:')
    print('  classifier.pth, localizer.pth, unet.pth')

FINAL CHECKPOINT STATUS
  ✅  classifier.pth         epoch=23  metric=0.8800  516MB
  ✅  localizer.pth          epoch=35  metric=0.6352  141MB
  ✅  unet_frozen.pth        epoch=24  metric=0.9205  82MB
  ✅  unet_partial.pth       epoch=21  metric=0.9251  82MB
  ✅  unet.pth               epoch=27  metric=0.9278  82MB

All checkpoints ready — download the 3 submission files:
  classifier.pth, localizer.pth, unet.pth


---
## CELL 11 — Test MultiTask model

In [ ]:
import torch, sys
sys.path.insert(0, '/content/drive/MyDrive/DA6401_Assignment2/code')

import os
CKPT = '/content/drive/MyDrive/DA6401_Assignment2/checkpoints'

from models.multitask import MultiTaskPerceptionModel
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MultiTaskPerceptionModel(
    classifier_path=os.path.join(CKPT, 'classifier.pth'),
    localizer_path =os.path.join(CKPT, 'localizer.pth'),
    unet_path      =os.path.join(CKPT, 'unet.pth'),
).to(device)
model.eval()

dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)

print('MultiTask forward pass:')
print(f'  classification : {out["classification"].shape}')   # [2,37]
print(f'  localization   : {out["localization"].shape}')     # [2,4]
print(f'  segmentation   : {out["segmentation"].shape}')     # [2,3,224,224]
print('\n✅ Pipeline working!')

ModuleNotFoundError: No module named 'multitask'

---
## CELL 12 — Download checkpoints to laptop

In [ ]:
from google.colab import files
import os
CKPT = '/content/drive/MyDrive/DA6401_Assignment2/checkpoints'

# Download the 3 mandatory submission files
for fname in ['classifier.pth', 'localizer.pth', 'unet.pth']:
    fp = os.path.join(CKPT, fname)
    if os.path.exists(fp):
        print(f'Downloading {fname} ...')
        files.download(fp)
    else:
        print(f'Skipping {fname} — not found')